In [ ]:
import numpy as np
import pandas as pd

class SimplexSolver:
    def __init__(self, C, A, b):
        self.C = np.array(C, dtype=float)  # Coefficients of the objective function
        self.A = np.array(A, dtype=float)  # Coefficients of constraints
        self.b = np.array(b, dtype=float)  # Right-hand side values
        self.num_vars = len(C)
        self.num_constraints = len(b)

        # Create initial simplex table
        self.table = np.hstack((self.A, np.eye(self.num_constraints), self.b.reshape(-1, 1)))
        self.CB = np.zeros(self.num_constraints)  # Coefficients of basic variables
        self.basic_vars = np.array([self.num_vars + i for i in range(self.num_constraints)])  # Initial basic variables

    def compute_Zj_Cj(self):
        """Calculate Zj and Cj-Zj."""
        Zj = np.dot(self.CB, self.table[:, :-1])  # Dot product of CB and table (excluding RHS column)
        Cj_Zj = self.C - Zj[:self.num_vars]  # Compute Cj - Zj for non-basic variables
        return Zj, Cj_Zj

    def find_key_column(self, Cj_Zj):
        """Find the entering variable (column with max Cj-Zj)."""
        key_col_index = np.argmax(Cj_Zj)  # Maximum positive value
        return key_col_index if Cj_Zj[key_col_index] > 0 else None

    def find_key_row(self, key_col_index):
        """Find the key row using minimum positive ratio test."""
        ratios = np.full(self.num_constraints, np.inf)  # Initialize with infinity
        for i in range(self.num_constraints):
            if self.table[i, key_col_index] > 0:  # Avoid division by zero
                ratios[i] = self.table[i, -1] / self.table[i, key_col_index]

        key_row_index = np.argmin(ratios)  # Least ratio
        return key_row_index if ratios[key_row_index] != np.inf else None

    def update_table(self, key_row, key_col):
        """Perform row operations to update the simplex table."""
        key_element = self.table[key_row, key_col]
        self.table[key_row, :] /= key_element  # Divide key row by pivot

        # Update remaining rows
        for i in range(self.num_constraints):
            if i != key_row:
                factor = self.table[i, key_col]
                self.table[i, :] -= factor * self.table[key_row, :]

        # Update CB and basic variables
        self.CB[key_row] = self.C[key_col]
        self.basic_vars[key_row] = key_col

    def solve(self):
        """Execute the Simplex Algorithm and print each iteration."""
        iteration = 1
        while True:
            print(f"\nIteration {iteration}: Simplex Table")
            df = pd.DataFrame(self.table, columns=[f'X{i+1}' for i in range(self.num_vars)] +
                              [f'S{i+1}' for i in range(self.num_constraints)] + ['Solution'])
            print(df)

            Zj, Cj_Zj = self.compute_Zj_Cj()
            print("\nZj:", Zj)
            print("Cj - Zj:", Cj_Zj)

            key_col = self.find_key_column(Cj_Zj)
            if key_col is None:  # Stop if optimal solution reached
                break

            key_row = self.find_key_row(key_col)
            if key_row is None:
                raise ValueError("Unbounded solution detected.")

            print(f"\nKey Column: X{key_col+1}, Key Row: {key_row+1}")
            self.update_table(key_row, key_col)
            iteration += 1

        # Final optimal solution
        print("\nFinal Optimal Solution:")
        final_solution = np.zeros(self.num_vars)
        for i in range(self.num_constraints):
            if self.basic_vars[i] < self.num_vars:
                final_solution[self.basic_vars[i]] = self.table[i, -1]

        print(pd.DataFrame([final_solution], columns=[f'X{i+1}' for i in range(self.num_vars)]))
        print(f"Optimal Value (Z): {np.dot(self.CB, self.table[:, -1])}")

# Example Usage
C = [12, 16, 1, 0]  # Coefficients of Objective Function
A = [[10, 20, 1, 0],  # Constraints Coefficients
     [8,  8,  0, 1]]
b = [120, 80]  # RHS values

solver = SimplexSolver(C, A, b)
solver.solve()
